# M2 — G1 đi bộ trên mặt phẳng 
Train M2 (MLP 896-448-128, input `o_t`) bằng MuJoCo Playground + Brax PPO trên Kaggle (2×T4).

- Môi trường: địa hình `flat` + cấu hình của Minh (`configs/terrain.yaml`, `UnifiedG1TerrainEnv`) — **không sửa**.
- Phần thêm mới nằm trong `flat_walking/` (xem `flat_walking/readme_report.md`).
- Chạy lần lượt từ Cell 1. Cell 5 là kiểm tra nhanh, nên chạy trước Cell 6 (train).

In [8]:
# Cell 1: Kiểm tra GPU
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB



In [9]:
# Cell 2: Clone / cập nhật repo (nhánh M2)
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")
if not PROJECT_DIR.exists():
    subprocess.check_call(["git", "clone", "--branch", "M2",
                           "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git", str(PROJECT_DIR)])
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %an %s"], capture_output=True, text=True).stdout)

Updating 9c3bac4..4fc7f52
Fast-forward
 .gitignore                        |   5 -
 configs/ppo_flat.yaml             |  58 -----
 flat_walking/g1_minh_env.py       | 147 +++++++++++++
 flat_walking/m2_flat.yaml         |  49 +++++
 flat_walking/m2_flat_kaggle.ipynb | 435 ++++++++++++++++++++++++++++++++++++++
 flat_walking/readme_report.md     | 170 +++++++++++++++
 flat_walking/train_m2_flat.py     | 322 ++++++++++++++++++++++++++++
 kaggle_remote.ipynb               | 358 -------------------------------
 readme_report.md                  |  88 --------
 readme_report_vi.md               |  88 --------
 scripts/backup_results.py         | 229 --------------------
 scripts/benchmark_gpu.py          |  20 +-
 scripts/extract_from_notebook.py  |  48 -----
 scripts/render_demo.py            | 201 ------------------
 scripts/train_flat.py             | 362 -------------------------------
 15 files changed, 1127 insertions(+), 1453 deletions(-)
 delete mode 100644 configs/ppo_flat.yaml
 cre

From https://github.com/TotallyNotMinh/Bipedal-Locomotion
   9c3bac4..4fc7f52  M2         -> origin/M2


In [10]:
# Cell 3: Cài dependencies
import sys, subprocess

pkgs = ["brax", "flax", "optax", "orbax-checkpoint", "ml_collections", "pyyaml",
        "lxml", "etils", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

try:
    import mujoco_playground
    print("[OK] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"])
    print("[OK] mujoco_playground installed")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[OK] mujoco_playground already installed


In [11]:
# Cell 4: Kiểm tra JAX / GPU
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
import jax
print(f"JAX {jax.__version__}:", jax.devices())

JAX 0.11.1: [CudaDevice(id=0), CudaDevice(id=1)]


In [13]:
# Cell 5: Kiểm tra nhanh môi trường
import subprocess, sys
r = subprocess.run([sys.executable, "flat_walking/train_m2_flat.py", "--smoke"],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1500:] if r.returncode else "")
assert r.returncode == 0, "smoke test lỗi — đừng chạy Cell 6"

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[smoke] Minh terrain: flat d=0.0 80.0x80.0 m, grid=(1000, 1000), height=[0.000, 0.000]
[smoke] nu=29 nhfield=1 hfield_size=[40.0, 40.0, 0.01, 0.05] floor_geom_type=mjGEOM_HFIELD
[smoke] obs_size={'privileged_state': (216,), 'state': (103,)}
[smoke] spawn xyz=(42.76, 23.48, 0.755)  (arena 80.0x80.0)  command=[ 0.57979214 -0.20242482 -0.05027163]
[smoke] sau 20 bước: z=0.698 reward=-0.0091 done=0.0
[smoke] OK




In [14]:
# Cell 6: TRAIN M2 (100M bước, ~vài chục phút -> vài giờ tuỳ GPU)
import subprocess, sys

NUM_TIMESTEPS = None     # None = lấy từ flat_walking/m2_flat.yaml (100M). Thử nhanh: 2_000_000
cmd = [sys.executable, "-u", "flat_walking/train_m2_flat.py"]
if NUM_TIMESTEPS:
    cmd += ["--num-timesteps", str(NUM_TIMESTEPS)]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "train lỗi"

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[+] JAX devices (2): ['cuda:0', 'cuda:1']
[+] Minh env: category=flat d=0.0 arena=80.0x80.0m grid=(1000, 1000) num_envs=4096
[+] Minh commands: vx=(0.2, 1.0) vy=(-0.3, 0.3) wz=(-0.5, 0.5)  push=False friction_switch=False
[+] obs_size={'privileged_state': (216,), 'state': (103,)} action_size=29 episode_length=1000
[+] M2 params: {'policy_input_dim': 103, 'value_input_dim': 216, 'policy_params': 559994, 'value_params': 275457}
[+] Output: /kaggle/working/Bipedal-Locomotion/results/flat/g1_flat_m2_20261126_083748
[+] Starting PPO training...
  step=          0  elapsed=   473s  sps=        0  eval_reward= -3.178  eval_len=  52.5
  step=  5,406,720  elapsed=  1548s  sps=    3,492  eval_reward= -1.381  eval_len=  57.6
  step= 10,813,440  elapsed=  2319s  sps=    4,663  eval_reward= -1.028  eval_len=  85.7
  step= 16,220,160  elapsed=  3008s  sps=    5,392  eval_reward=  0.538  eval_len= 167.4

In [15]:
# Cell 7: Nén kết quả để tải về (results/flat -> /kaggle/working/m2_flat_results.zip)
import shutil
print(shutil.make_archive("/kaggle/working/m2_flat_results", "zip", "results/flat"))

/kaggle/working/m2_flat_results.zip


## Đánh giá kết quả train
Cell 8: kiểm tra + sao lưu checkpoint. Cell 9: phân tích đường cong train. Cell 10: rollout policy đã lưu, đo khả năng bám lệnh vận tốc. Logic nằm trong `flat_walking/eval_m2_flat.py`.

In [16]:
# Cell 8: Kiểm tra checkpoint đã lưu chưa + sao lưu (/kaggle/working/m2_flat_backup)
import subprocess, sys
r = subprocess.run([sys.executable, 'flat_walking/eval_m2_flat.py', '--check',
                    '--backup-dir', '/kaggle/working/m2_flat_backup'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else '')


/usr/bin/python3: can't open file '/kaggle/working/Bipedal-Locomotion/flat_walking/eval_m2_flat.py': [Errno 2] No such file or directory



In [17]:
# Cell 9: Phân tích đường cong train (reward, độ dài episode, hội tụ, thành phần reward)
r = subprocess.run([sys.executable, 'flat_walking/eval_m2_flat.py', '--curves'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else '')
import glob; from IPython.display import Image, display
for p in glob.glob('results/flat/*/train_curves.png')[-1:]: display(Image(p))


/usr/bin/python3: can't open file '/kaggle/working/Bipedal-Locomotion/flat_walking/eval_m2_flat.py': [Errno 2] No such file or directory



In [18]:
# Cell 10: Rollout policy từ checkpoint cuối: sống sót, sai số bám vận tốc (vx, vy, wz)
r = subprocess.run([sys.executable, 'flat_walking/eval_m2_flat.py', '--rollout', '--n-envs', '64'],
                   capture_output=True, text=True)
print(r.stdout[-5000:]); print(r.stderr[-3000:] if r.returncode else '')


/usr/bin/python3: can't open file '/kaggle/working/Bipedal-Locomotion/flat_walking/eval_m2_flat.py': [Errno 2] No such file or directory



In [19]:
# Cell 11: Nén lại lần nữa (gồm cả train_curves.png và eval_rollout.json)
import shutil
print(shutil.make_archive('/kaggle/working/m2_flat_results', 'zip', 'results/flat'))

/kaggle/working/m2_flat_results.zip
